# Model Testing

In [ ]:
import sys
sys.path.append('../')

In [ ]:
import os
import torch

from tqdm import tqdm

from sklearn.metrics import accuracy_score, f1_score, confusion_matrix

from monai.networks.nets import DenseNet
from torch.utils.data import DataLoader

from src.data.dataset import LABELS, BrainMriDataset
from src.data.transforms import Transforms
from src.utils.model import load_model
from src.utils.visualisation import plot_confusion_matrix

In [ ]:
DEVICE      = 'cuda' if torch.cuda.is_available() else 'cpu'
TEST_CSV    = '../data/processed/test.csv'
IMAGES_PATH = '../data/images/ADNI/derivatives'
INPUT_PATH  = '../models/'
LOGS_PATH   = '../logs/'
NUM_WORKERS = 5
BATCH_SIZE  = 16

In [ ]:
assert os.path.exists(TEST_CSV)
assert os.path.exists(IMAGES_PATH)
assert os.path.exists(INPUT_PATH)

In [ ]:
test_transform = Transforms.get_data_loading()

test_dataset = BrainMriDataset(
    csv_path=TEST_CSV,
    images_path=IMAGES_PATH,
    transform=test_transform
)

test_loader = DataLoader(
    dataset=test_dataset,
    batch_size=BATCH_SIZE,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    shuffle=False
)

In [ ]:
model = DenseNet(spatial_dims=3, in_channels=1, out_channels=1, dropout_prob=0.2).to(DEVICE)
load_model(model, INPUT_PATH, DEVICE)

In [ ]:
y_true = []
y_pred = []
model.eval()

with torch.no_grad():
    for step, batch in tqdm(enumerate(test_loader), 'Testing', len(test_loader)):
        with torch.autocast(DEVICE):
            labels = batch['label'].to(DEVICE).float().unsqueeze(1)
            images = batch['image'].to(DEVICE)

        y_pred_prob = torch.sigmoid(model(images))
        y_pred_label = (y_pred_prob > 0.5).float()

        y_true.extend(labels.cpu().numpy())
        y_pred.extend(y_pred_label.cpu().numpy())

In [ ]:
accuracy = accuracy_score(y_true, y_pred)
fscore = f1_score(y_true, y_pred)
cmatrix = confusion_matrix(y_true, y_pred)
class_names = LABELS.keys()

print('Accuracy-Score: {}'.format(accuracy))
print('F1-Score: {}'.format(fscore))
plot_confusion_matrix(cmatrix, class_names, figsize=(5, 5))